# 13. #06 파라미터 기반 시드 앙상블 비교

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import os

RANDOM_STATE = 42

tuned_params_06 = {
    'n_estimators': 5000,
    'learning_rate': 0.03,
    'num_leaves': 127,
    'min_child_samples': 10,
    'verbose': -1,
}

SEEDS_5 = [42, 43, 44, 45, 46]
SEEDS_10 = list(range(42, 52))


## 1. Data Load

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

train = train.drop_duplicates(subset=[col for col in train.columns if col != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

print('train:', train.shape, '/ test:', test.shape)


train: (2994, 18) / test: (3000, 17)


## 2. 파생변수 생성

In [3]:
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)

    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']

    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data


train = add_features(train)
test = add_features(test)
print('파생변수 적용 후 train shape:', train.shape)


파생변수 적용 후 train shape: (2994, 37)


## 3. 인코딩

In [4]:
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])

    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

print('x_train:', x_train.shape, '/ x_test:', x_test.shape)


x_train: (2994, 35) / x_test: (3000, 35)


## 4. 시드 앙상블 함수

In [5]:
def run_seed_ensemble(seeds, params):
    seed_oof_list = []
    seed_test_list = []

    for seed in seeds:
        kf = KFold(n_splits=5, shuffle=True, random_state=seed)
        seed_oof = np.zeros(len(x_train))
        seed_test = np.zeros(len(x_test))

        model_params = dict(params)
        model_params['random_state'] = seed

        for tr_idx, va_idx in kf.split(x_train):
            X_tr, X_va = x_train.iloc[tr_idx], x_train.iloc[va_idx]
            y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

            model = LGBMRegressor(**model_params)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])

            seed_oof[va_idx] = model.predict(X_va)
            seed_test += model.predict(x_test) / kf.n_splits

        seed_oof_list.append(seed_oof)
        seed_test_list.append(seed_test)
        print(f'  seed={seed} MAE = {mean_absolute_error(y_train, seed_oof):.4f}')

    oof_matrix = np.column_stack(seed_oof_list)
    test_matrix = np.column_stack(seed_test_list)
    avg_oof = oof_matrix.mean(axis=1)
    avg_test = test_matrix.mean(axis=1)
    mae = mean_absolute_error(y_train, avg_oof)
    return mae, avg_test, oof_matrix


## 5. 단일모델 (seed=42)

In [6]:
mae_single06, test_single06, _ = run_seed_ensemble([42], tuned_params_06)
print(f'단일모델 MAE = {mae_single06:.4f}')


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=42 MAE = 0.1740
단일모델 MAE = 0.1740


## 6. 5-seed 앙상블

In [7]:
mae_5seed, test_5seed, oof_5seed = run_seed_ensemble(SEEDS_5, tuned_params_06)
print(f'5-seed 평균 MAE = {mae_5seed:.4f}')


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=42 MAE = 0.1740


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=43 MAE = 0.1721


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=44 MAE = 0.1783


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=45 MAE = 0.1759


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=46 MAE = 0.1742
5-seed 평균 MAE = 0.1717


## 7. 10-seed 앙상블

In [8]:
mae_10seed, test_10seed, oof_10seed = run_seed_ensemble(SEEDS_10, tuned_params_06)
print(f'10-seed 평균 MAE = {mae_10seed:.4f}')


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=42 MAE = 0.1740


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=43 MAE = 0.1721


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

...

  seed=49 MAE = 0.1746


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=50 MAE = 0.1742


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

  seed=51 MAE = 0.1752
10-seed 평균 MAE = 0.1709


## 8. 비교

In [9]:
print(f'단일모델      : {mae_single06:.4f}')
print(f'5-seed 앙상블 : {mae_5seed:.4f}')
print(f'10-seed 앙상블: {mae_10seed:.4f}')


단일모델      : 0.1740
5-seed 앙상블 : 0.1717
10-seed 앙상블: 0.1709


## 9. 제출 파일 저장

In [10]:
os.makedirs('../submissions', exist_ok=True)

results = {
    'single06': (mae_single06, test_single06, 'submit_13_single06.csv'),
    '5seed06': (mae_5seed, test_5seed, 'submit_13_5seed06.csv'),
    '10seed06': (mae_10seed, test_10seed, 'submit_13_10seed06.csv'),
}

best_name = min(results, key=lambda k: results[k][0])
best_mae, best_test, best_fname = results[best_name]
print(f'{best_name}: {best_mae:.4f}')

final_pred_13 = np.clip(best_test, 0, 1)
sample_submission['stress_score'] = final_pred_13
sample_submission.to_csv(f'../submissions/{best_fname}', index=False)
sample_submission.head()


10seed06: 0.1709


,ID,stress_score
0,TEST_0000,0.543689
1,TEST_0001,0.851889
2,TEST_0002,0.306827
3,TEST_0003,0.502750
4,TEST_0004,0.557616
